In [1]:
import torch
import torch.nn as nn

from src.util.processed_dataset import ProcessedDataset
from src.util.load_dataset import LoadDataset
from src.model.lstm_model import LSTMModel
from src.model.lstm_train import LSTMTrain
from src.model.lstm_predict import LSTMPredict
from src.model.lstm_eval import LSTMEval
from src.model.lstm_rouge import LSTMRouge
from src.model.distilgpt2_model import Distilgpt2Model
from transformers import AutoTokenizer


/home/ramis/practicum/text_autocomplete/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdt = ProcessedDataset(
        path = "text", 
        data_files="raw_dataset.txt", 
        clean_path="processed_dataset.csv", 
        train_path="train_dataset.csv",
        val_path="val_dataset.csv",
        test_path="test_dataset.csv",
        split="train[:100000]"
    )
pdt.process()

Creating CSV from Arrow format: 100%|██████████| 100/100 [00:00<00:00, 484.20ba/s]


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
batch_size = 64

load_dataset = LoadDataset(tokenizer=bert_tokenizer, batch_size=batch_size, max_len=8)
train_loader = load_dataset.getDataLoader("train_dataset.csv", shuffle=True)
val_loader = load_dataset.getDataLoader("val_dataset.csv", shuffle=False)

print("train_loader:", len(train_loader))
print("val_loader:", len(val_loader))


train_loader: 18114
val_loader: 2281


In [4]:
lstm_model = LSTMModel(bert_tokenizer.vocab_size).to(device)
adam_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.002)
cross_entropy_loss = nn.CrossEntropyLoss()

In [5]:
lstm_train = LSTMTrain(model=lstm_model, loader=train_loader, device=device)
lstm_eval = LSTMEval(model=lstm_model, loader=val_loader, device=device)

epoch_val = 3
for epoch in range(epoch_val):
    loss = lstm_train.train(optimizer=adam_optimizer, criterion=cross_entropy_loss, tokenizer=bert_tokenizer)  
    accuracy = lstm_eval.evaluate()
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Accuracy = {accuracy:.4f}") 

100%|██████████| 18114/18114 [02:26<00:00, 123.97it/s]


Epoch 1: Loss = 6.3818, Accuracy = 0.1454


100%|██████████| 18114/18114 [02:25<00:00, 124.79it/s]


Epoch 2: Loss = 5.9308, Accuracy = 0.1572


100%|██████████| 18114/18114 [02:26<00:00, 123.99it/s]


Epoch 3: Loss = 5.8002, Accuracy = 0.1640


In [6]:
lstm_eval = LSTMRouge(model=lstm_model, tokenizer=bert_tokenizer, device=device)

test_text = load_dataset.getText("test_dataset.csv")

results = lstm_eval.get_metrics(test_text)

print('Metrics')
for k, v in results.items():
    print(f"{k}: {v:.4f}")
     

Metrics
rouge1: 0.9072
rouge2: 0.8907
rougeL: 0.9072
rougeLsum: 0.9073


In [7]:
examples = [
    "Hello, my name is",
    "Today",
    "How are you doing",
]
max_length = 10
lstm_predict = LSTMPredict(model=lstm_model, tokenizer=bert_tokenizer, device=device, max_length=max_length)

for text in examples:
    print(lstm_predict.generate(text))


Hello, my name is so much i can ##t ##wee ##t ##wee is ##m
Today i want to go to bed but i have to
How are you doing you guys are you so much so much i can


In [8]:
max_length = 10
distilgpt2_model = Distilgpt2Model(max_length=max_length)
examples = [
    "Hello, my name is",
    "Today",
    "How are you doing",
]

with torch.no_grad():
    for text in examples:
        predict = distilgpt2_model.generate(text)
        print(predict)

Device set to use cuda:0


Hello, my name is Hélène, and I have a
TodayInventing a new Internet of Things, and
How are you doing the study?







